In [1]:
import pandas as pd
import numpy as np

def normalize_database(input_csv="database.csv", output_csv="database_normalized.csv"):
    # 1. Load the data
    df = pd.read_csv(input_csv)

    # 2. Define columns
    metadata_cols = ['GSE_ID', 'GSM_ID', 'GPL', 'Disease_State', 'Data_Type', 'title']
    gene_cols = [col for col in df.columns if col not in metadata_cols]

    # 3. Handle missing values
    # Temporarily replace -1.0 with NaN so they don't ruin the math
    df[gene_cols] = df[gene_cols].replace(-1.0, np.nan)

    # 4. Apply Z-score Normalization WITHIN each GSE Series
    # This is more robust than global normalization because it accounts for
    # different tech/scaling used by different researchers.
    def zscore(x):
        return (x - x.mean()) / x.std()

    # Group by GSE_ID and apply the zscore function to each gene column
    df_norm = df.groupby('GSE_ID')[gene_cols].transform(zscore)

    # 5. Post-processing
    # Fill any NaNs created by the math (e.g., if std was 0) or the original -1s
    # We fill with 0 because in Z-score space, 0 represents the average expression
    df_norm = df_norm.fillna(0)

    # 6. Combine back with metadata
    final_df = pd.concat([df[metadata_cols], df_norm], axis=1)

    # 7. Save output
    final_df.to_csv(output_csv, index=False)
    print(f"Normalization complete! File saved as: {output_csv}")

    return final_df

# Execute
normalized_data = normalize_database("/content/database_robust.csv")

Normalization complete! File saved as: database_normalized.csv
